In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()

candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    cwd / "deployment_models",
    cwd / "code files" / "ml" / "deployment_models",
    cwd.parent / "deployment_models",
    cwd.parent / "ml" / "deployment_models",
    cwd.parent / "code files" / "ml" / "deployment_models",
    Path(r"C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models"),
]

DEPLOYMENT_DIR = None

for candidate in candidates:
    if (candidate / "occupancy_deployment_model_v2.py").exists():
        DEPLOYMENT_DIR = candidate
        break

if DEPLOYMENT_DIR is None:
    print("Current working directory:", cwd)
    print("\nChecked these folders:")
    for candidate in candidates:
        print(candidate)
    raise FileNotFoundError("Could not find occupancy_deployment_model_v2.py")

ML_DIR = DEPLOYMENT_DIR.parent

for path in [DEPLOYMENT_DIR, ML_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print("DEPLOYMENT_DIR:", DEPLOYMENT_DIR)
print("ML_DIR:", ML_DIR)
print("File exists:", (DEPLOYMENT_DIR / "occupancy_deployment_model_v2.py").exists())

DEPLOYMENT_DIR: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models
ML_DIR: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml
File exists: True


In [3]:
from gold_ml_modeling import (
    OCCUPANCY_AVAILABILITY_FEATURES,
    OCCUPANCY_BASE_FEATURES,
    build_mysql_engine,
    load_property_mart_frame,
    prepare_common_features,
)

from occupancy_deployment_model_v2 import (
    PRICE_GAP_FEATURES,
    SEED,
    add_price_gap_features,
    build_bucket_metrics,
    choose_best_occupancy_model,
    fit_predict_occupancy,
    make_oof_fair_price_predictions,
    occupancy_metric_bundle,
    predict_xgb_hgb_blend,
    tune_xgb_hgb_blend,
)

In [4]:
engine = build_mysql_engine()
model_df = load_property_mart_frame(engine)

print(model_df.shape)
model_df.head()

(10005, 58)


,property_id,property_key,host_key,snapshot_date,city,region_group,market_segment,market_type,neighbourhood_cleansed,neighbourhood_group_cleansed,...,financing_vintage_year,applied_asset_discount_pct,applied_annual_interest_rate,host_since,host_response_rate,host_acceptance_rate,host_listings_count,host_total_listings_count,target_occupancy_rate,target_nightly_price
0,29396,3,5,2026-05-07,lisbon,lisbon_core,city_area,urban,Santa Maria Maior,Lisboa,...,2021,0.22,0.0185,2010-05-17,100.0,100.0,1.0,1.0,0.526027,77.0
1,83700,15,66,2026-05-07,lisbon,lisbon_coastal,coast_beach,beach,Cascais e Estoril,Cascais,...,2021,0.22,0.0185,2011-03-22,100.0,100.0,2.0,2.0,0.641096,87.0
2,89015,19,72,2026-05-07,lisbon,lisbon_core,city_area,urban,Misericrdia,Lisboa,...,2021,0.22,0.0185,2011-04-05,100.0,100.0,1.0,1.0,0.476712,131.0
3,119120,31,52,2026-05-07,lisbon,lisbon_core,city_area,urban,Estrela,Lisboa,...,2021,0.22,0.0185,2011-02-11,100.0,100.0,4.0,4.0,0.526027,125.0
4,122998,34,109,2026-05-07,lisbon,lisbon_core,city_area,urban,Misericrdia,Lisboa,...,2021,0.22,0.0185,2011-05-23,100.0,100.0,1.0,2.0,0.698630,104.0


In [5]:
prepared_df = (
    prepare_common_features(model_df)
    .dropna(subset=["target_occupancy_rate", "target_nightly_price"])
    .copy()
)

print(prepared_df.shape)
prepared_df[["property_id", "target_occupancy_rate", "target_nightly_price"]].head()

(10005, 71)


,property_id,target_occupancy_rate,target_nightly_price
0,29396,0.526027,77.0
1,83700,0.641096,87.0
2,89015,0.476712,131.0
3,119120,0.526027,125.0
4,122998,0.698630,104.0


In [6]:
baseline_features = OCCUPANCY_BASE_FEATURES + OCCUPANCY_AVAILABILITY_FEATURES
price_gap_features = baseline_features + PRICE_GAP_FEATURES

missing_base_features = [col for col in baseline_features if col not in prepared_df.columns]

if missing_base_features:
    raise ValueError(f"Missing baseline features: {missing_base_features}")

print("Baseline features available.")
print("Baseline feature count:", len(baseline_features))
print("Price-gap feature count:", len(price_gap_features))

Baseline features available.
Baseline feature count: 47
Price-gap feature count: 52


In [7]:
X = prepared_df.drop(columns=["target_occupancy_rate"]).reset_index(drop=True)
y_occ = prepared_df["target_occupancy_rate"].reset_index(drop=True)
y_price = prepared_df["target_nightly_price"].reset_index(drop=True)

X_train, X_test, y_occ_train, y_occ_test, y_price_train, y_price_test = train_test_split(
    X,
    y_occ,
    y_price,
    test_size=0.2,
    random_state=SEED,
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_occ_train = y_occ_train.reset_index(drop=True)
y_occ_test = y_occ_test.reset_index(drop=True)
y_price_train = y_price_train.reset_index(drop=True)
y_price_test = y_price_test.reset_index(drop=True)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (8004, 70)
X_test: (2001, 70)


In [8]:
fair_price_train_pred, fair_price_test_pred, fair_price_fold_details, fair_price_test_detail = (
    make_oof_fair_price_predictions(
        X_train,
        y_price_train,
        X_test,
        y_price_test,
    )
)

fair_price_fold_df = pd.DataFrame(fair_price_fold_details)
fair_price_fold_df

,fold,holdout_rows,fair_price_r2,fair_price_rmse,fair_price_mae,chosen_blend_strategy
0,1,2668,0.666866,41.688433,28.190388,"{'strategy': 'segment_specific_weight', 'weigh..."
1,2,2668,0.684714,41.941143,28.267650,"{'strategy': 'bucket_specific_weight', 'weight..."
2,3,2668,0.654947,42.210433,28.945690,"{'strategy': 'bucket_specific_weight', 'weight..."


In [9]:
X_train_gap = add_price_gap_features(X_train, fair_price_train_pred)
X_test_gap = add_price_gap_features(X_test, fair_price_test_pred)

missing_gap_features = [col for col in price_gap_features if col not in X_train_gap.columns]

if missing_gap_features:
    raise ValueError(f"Missing price-gap features: {missing_gap_features}")

X_train_gap[PRICE_GAP_FEATURES].head()

,predicted_fair_price,price_gap_abs,price_gap_pct,overpriced_flag,underpriced_flag
0,109.019071,-18.019071,-0.165284,0,1
1,134.745819,-10.745819,-0.079749,0,0
2,133.145948,16.854052,0.126583,1,0
3,80.835548,-10.835548,-0.134044,0,1
4,69.835165,100.164835,1.434304,1,0


In [10]:
predictions = {}
fitted_models = {}
model_registry = {}

baseline_tests = [
    ("A_hgb_baseline", "hist_gradient_boosting", X_train, X_test, baseline_features, False),
    ("B_xgb_baseline", "xgboost", X_train, X_test, baseline_features, False),
    ("C_catboost_baseline", "catboost", X_train, X_test, baseline_features, False),
]

for variant, model_name, train_frame, test_frame, features, logit_target in baseline_tests:
    fitted_models[variant], predictions[variant] = fit_predict_occupancy(
        model_name=model_name,
        X_train=train_frame,
        y_train=y_occ_train,
        X_test=test_frame,
        feature_columns=features,
        logit_target=logit_target,
    )

    model_registry[variant] = {
        "model_type": "single_pipeline",
        "base_model": model_name,
        "feature_columns": features,
        "uses_price_gap": False,
        "logit_target": logit_target,
    }

pd.DataFrame(
    [
        {"model_variant": name, **occupancy_metric_bundle(y_occ_test, pred)}
        for name, pred in predictions.items()
    ]
).sort_values(["r2", "rmse", "mae"], ascending=[False, True, True])

,model_variant,r2,rmse,mae
0,A_hgb_baseline,0.734456,0.084069,0.059150
1,B_xgb_baseline,0.729949,0.084779,0.060141
2,C_catboost_baseline,0.722559,0.085931,0.062417


In [11]:
price_gap_tests = [
    ("D_hgb_plus_price_gap", "hist_gradient_boosting", X_train_gap, X_test_gap, price_gap_features, False),
    ("E_xgb_plus_price_gap", "xgboost", X_train_gap, X_test_gap, price_gap_features, False),
    ("F_catboost_plus_price_gap", "catboost", X_train_gap, X_test_gap, price_gap_features, False),
    ("G_xgb_logit_plus_price_gap", "xgboost", X_train_gap, X_test_gap, price_gap_features, True),
]

for variant, model_name, train_frame, test_frame, features, logit_target in price_gap_tests:
    fitted_models[variant], predictions[variant] = fit_predict_occupancy(
        model_name=model_name,
        X_train=train_frame,
        y_train=y_occ_train,
        X_test=test_frame,
        feature_columns=features,
        logit_target=logit_target,
    )

    model_registry[variant] = {
        "model_type": "single_pipeline",
        "base_model": model_name,
        "feature_columns": features,
        "uses_price_gap": True,
        "logit_target": logit_target,
    }

pd.DataFrame(
    [
        {"model_variant": name, **occupancy_metric_bundle(y_occ_test, pred)}
        for name, pred in predictions.items()
    ]
).sort_values(["r2", "rmse", "mae"], ascending=[False, True, True])

,model_variant,r2,rmse,mae
6,G_xgb_logit_plus_price_gap,0.735255,0.083942,0.058926
0,A_hgb_baseline,0.734456,0.084069,0.059150
3,D_hgb_plus_price_gap,0.733111,0.084282,0.059279
4,E_xgb_plus_price_gap,0.731436,0.084546,0.059921
1,B_xgb_baseline,0.729949,0.084779,0.060141
5,F_catboost_plus_price_gap,0.726001,0.085397,0.061769
2,C_catboost_baseline,0.722559,0.085931,0.062417


In [12]:
blend_tuning = tune_xgb_hgb_blend(
    X_train_gap,
    y_occ_train,
    price_gap_features,
)

blend_validation_df = (
    blend_tuning["validation_rows"]
    .sort_values(["rmse", "mae", "r2"], ascending=[True, True, False])
    .reset_index(drop=True)
)

blend_validation_df.head(10)

,weight_xgb,weight_hgb,r2,rmse,mae
0,0.625,0.375,0.742978,0.082312,0.058371
1,0.650,0.350,0.742977,0.082312,0.058397
2,0.600,0.400,0.742956,0.082315,0.058348
3,0.675,0.325,0.742954,0.082316,0.058427
4,0.575,0.425,0.742911,0.082323,0.058327
5,0.700,0.300,0.742908,0.082323,0.058458
6,0.550,0.450,0.742844,0.082333,0.058309
7,0.725,0.275,0.742840,0.082334,0.058491
8,0.525,0.475,0.742754,0.082348,0.058292
9,0.750,0.250,0.742749,0.082348,0.058531


In [13]:
blend_models, predictions["H_xgb_hgb_blend_plus_price_gap"] = predict_xgb_hgb_blend(
    X_train_gap,
    y_occ_train,
    X_test_gap,
    price_gap_features,
    blend_tuning["best_weight_xgb"],
)

fitted_models["H_xgb_hgb_blend_plus_price_gap"] = blend_models

model_registry["H_xgb_hgb_blend_plus_price_gap"] = {
    "model_type": "xgb_hgb_blend",
    "base_model": "xgboost_hist_gradient_boosting_blend",
    "feature_columns": price_gap_features,
    "uses_price_gap": True,
    "logit_target": False,
}

print("Best XGB weight:", blend_tuning["best_weight_xgb"])
print("Best HGB weight:", blend_tuning["best_weight_hgb"])

Best XGB weight: 0.625
Best HGB weight: 0.375


In [14]:
overall_df_raw = pd.DataFrame(
    [
        {
            "model_variant": model_name,
            **occupancy_metric_bundle(y_occ_test, pred),
        }
        for model_name, pred in predictions.items()
    ]
)

overall_df_raw

,model_variant,r2,rmse,mae
0,A_hgb_baseline,0.734456,0.084069,0.059150
1,B_xgb_baseline,0.729949,0.084779,0.060141
2,C_catboost_baseline,0.722559,0.085931,0.062417
3,D_hgb_plus_price_gap,0.733111,0.084282,0.059279
4,E_xgb_plus_price_gap,0.731436,0.084546,0.059921
5,F_catboost_plus_price_gap,0.726001,0.085397,0.061769
6,G_xgb_logit_plus_price_gap,0.735255,0.083942,0.058926
7,H_xgb_hgb_blend_plus_price_gap,0.735411,0.083917,0.059229


In [15]:
overall_df_raw = pd.DataFrame(
    [
        {
            "model_variant": model_name,
            **occupancy_metric_bundle(y_occ_test, pred),
        }
        for model_name, pred in predictions.items()
    ]
)

overall_df_raw

,model_variant,r2,rmse,mae
0,A_hgb_baseline,0.734456,0.084069,0.059150
1,B_xgb_baseline,0.729949,0.084779,0.060141
2,C_catboost_baseline,0.722559,0.085931,0.062417
3,D_hgb_plus_price_gap,0.733111,0.084282,0.059279
4,E_xgb_plus_price_gap,0.731436,0.084546,0.059921
5,F_catboost_plus_price_gap,0.726001,0.085397,0.061769
6,G_xgb_logit_plus_price_gap,0.735255,0.083942,0.058926
7,H_xgb_hgb_blend_plus_price_gap,0.735411,0.083917,0.059229


In [16]:
selection = choose_best_occupancy_model(overall_df_raw)

selected_model_name = selection["selected_model"]
ranked_overall_df = selection["ranked_metrics"]

print("Selected model:", selected_model_name)
print("R2:", selection["selected_r2"])
print("RMSE:", selection["selected_rmse"])
print("MAE:", selection["selected_mae"])

ranked_overall_df

Selected model: H_xgb_hgb_blend_plus_price_gap
R2: 0.7354114050569655
RMSE: 0.0839174538348435
MAE: 0.059228511604062214


,model_variant,r2,rmse,mae
0,H_xgb_hgb_blend_plus_price_gap,0.735411,0.083917,0.059229
1,G_xgb_logit_plus_price_gap,0.735255,0.083942,0.058926
2,A_hgb_baseline,0.734456,0.084069,0.059150
3,D_hgb_plus_price_gap,0.733111,0.084282,0.059279
4,E_xgb_plus_price_gap,0.731436,0.084546,0.059921
5,B_xgb_baseline,0.729949,0.084779,0.060141
6,F_catboost_plus_price_gap,0.726001,0.085397,0.061769
7,C_catboost_baseline,0.722559,0.085931,0.062417


In [17]:
bucket_df = build_bucket_metrics(
    y_occ_train,
    y_occ_test,
    predictions,
)

bucket_df

,occupancy_bucket,model_variant,rows,actual_mean_occupancy,r2,rmse,mae
16,low_occupancy,A_hgb_baseline,538,0.311208,-1.054937,0.102575,0.076354
17,low_occupancy,B_xgb_baseline,538,0.311208,-1.080275,0.103206,0.077214
18,low_occupancy,C_catboost_baseline,538,0.311208,-1.170008,0.105408,0.080809
19,low_occupancy,D_hgb_plus_price_gap,538,0.311208,-1.065582,0.102841,0.076288
20,low_occupancy,E_xgb_plus_price_gap,538,0.311208,-1.088063,0.103399,0.077060
21,low_occupancy,F_catboost_plus_price_gap,538,0.311208,-1.146797,0.104843,0.080656
22,low_occupancy,G_xgb_logit_plus_price_gap,538,0.311208,-1.018923,0.101672,0.074910
23,low_occupancy,H_xgb_hgb_blend_plus_price_gap,538,0.311208,-1.059589,0.102691,0.076316
0,medium_high_occupancy,A_hgb_baseline,980,0.680951,-4.207622,0.074951,0.049310
1,medium_high_occupancy,B_xgb_baseline,980,0.680951,-4.360821,0.076046,0.050388


In [18]:
fair_price_test_detail

{'price_model_version': 'nightly_price_deployment_model_v2',
 'chosen_blend_strategy': {'strategy': 'bucket_specific_weight',
  'weight_scope': 'predicted_price_bucket_from_baseline',
  'global_fallback_weight': 0.86,
  'r2': 0.6861477252369117,
  'rmse': 41.65840660134132,
  'mae': 28.73810852128084,
  'mape': 0.20007459296319607,
  'residual_mean': -3.8113066105001776,
  'residual_std': 41.48369297065337},
 'segmented_config': {'threshold_q': 0.925,
  'threshold_value': 266.0,
  'target_mode': 'raw_price',
  'normal_model': 'xgboost',
  'luxury_model': 'ridge',
  'route': 'soft_route',
  'r2': 0.6844449298931793,
  'rmse': 41.77126190257283,
  'mae': 28.90869225457689,
  'mape': 0.20356248146769615,
  'residual_mean': -2.789467168310093,
  'residual_std': 41.67801811327229,
  'luxury_train_rows': 387},
 'fair_price_metrics': {'r2': 0.7047989216300299,
  'rmse': 39.76767473414737,
  'mae': 26.958381197251715,
  'mape': 0.1954672668276793,
  'residual_mean': 0.9268223384338207,
  'resi

In [19]:
print("OCCUPANCY CLEAN TEST COMPLETE")
print()
print("Selected model:", selected_model_name)
print("Selected R2:", selection["selected_r2"])
print("Selected RMSE:", selection["selected_rmse"])
print("Selected MAE:", selection["selected_mae"])
print("Fair price model version: nightly_price_deployment_model_v2")

ranked_overall_df

OCCUPANCY CLEAN TEST COMPLETE

Selected model: H_xgb_hgb_blend_plus_price_gap
Selected R2: 0.7354114050569655
Selected RMSE: 0.0839174538348435
Selected MAE: 0.059228511604062214
Fair price model version: nightly_price_deployment_model_v2


,model_variant,r2,rmse,mae
0,H_xgb_hgb_blend_plus_price_gap,0.735411,0.083917,0.059229
1,G_xgb_logit_plus_price_gap,0.735255,0.083942,0.058926
2,A_hgb_baseline,0.734456,0.084069,0.059150
3,D_hgb_plus_price_gap,0.733111,0.084282,0.059279
4,E_xgb_plus_price_gap,0.731436,0.084546,0.059921
5,B_xgb_baseline,0.729949,0.084779,0.060141
6,F_catboost_plus_price_gap,0.726001,0.085397,0.061769
7,C_catboost_baseline,0.722559,0.085931,0.062417


In [20]:
from itertools import combinations
from sklearn.model_selection import train_test_split

def tune_blend_weights_on_validation(
    candidate_defs,
    y_train,
    y_valid,
    weight_grid=None,
):
    if weight_grid is None:
        weight_grid = [round(x, 2) for x in np.linspace(0.0, 1.0, 21)]

    rows = []
    best = None

    if len(candidate_defs) == 2:
        a, b = candidate_defs

        for w_a in weight_grid:
            w_b = 1 - w_a
            pred = w_a * a["valid_pred"] + w_b * b["valid_pred"]
            metrics = occupancy_metric_bundle(y_valid, pred)

            row = {
                "blend_name": f"blend_{a['variant']}__{b['variant']}",
                "model_count": 2,
                "components": [a["variant"], b["variant"]],
                "weights": {a["variant"]: float(w_a), b["variant"]: float(w_b)},
                **metrics,
            }
            rows.append(row)

            if best is None or (row["rmse"], row["mae"], -row["r2"]) < (
                best["rmse"], best["mae"], -best["r2"]
            ):
                best = row

    elif len(candidate_defs) == 3:
        a, b, c = candidate_defs

        for w_a in weight_grid:
            for w_b in weight_grid:
                w_c = 1 - w_a - w_b

                if w_c < 0 or w_c > 1:
                    continue

                pred = (
                    w_a * a["valid_pred"]
                    + w_b * b["valid_pred"]
                    + w_c * c["valid_pred"]
                )
                metrics = occupancy_metric_bundle(y_valid, pred)

                row = {
                    "blend_name": f"blend_{a['variant']}__{b['variant']}__{c['variant']}",
                    "model_count": 3,
                    "components": [a["variant"], b["variant"], c["variant"]],
                    "weights": {
                        a["variant"]: float(w_a),
                        b["variant"]: float(w_b),
                        c["variant"]: float(w_c),
                    },
                    **metrics,
                }
                rows.append(row)

                if best is None or (row["rmse"], row["mae"], -row["r2"]) < (
                    best["rmse"], best["mae"], -best["r2"]
                ):
                    best = row

    else:
        raise ValueError("Only top-2 and top-3 blends are supported.")

    return pd.DataFrame(rows), best


# Split training data again for blend-weight tuning only
X_blend_train, X_blend_valid, y_blend_train, y_blend_valid = train_test_split(
    X_train_gap,
    y_occ_train,
    test_size=0.2,
    random_state=SEED,
)

# Candidate definitions for validation tuning
blend_candidate_defs = []

# Use the current ranked candidates, but train validation versions
candidate_for_blend_tests = [
    ("A_hgb_baseline", "hist_gradient_boosting", X_train, baseline_features, False, False),
    ("B_xgb_baseline", "xgboost", X_train, baseline_features, False, False),
    ("C_catboost_baseline", "catboost", X_train, baseline_features, False, False),
    ("D_hgb_plus_price_gap", "hist_gradient_boosting", X_train_gap, price_gap_features, False, True),
    ("E_xgb_plus_price_gap", "xgboost", X_train_gap, price_gap_features, False, True),
    ("F_catboost_plus_price_gap", "catboost", X_train_gap, price_gap_features, False, True),
    ("G_xgb_logit_plus_price_gap", "xgboost", X_train_gap, price_gap_features, True, True),
]

validation_predictions = {}

for variant, model_name, source_frame, features, logit_target, uses_gap in candidate_for_blend_tests:
    if uses_gap:
        train_frame = X_blend_train
        valid_frame = X_blend_valid
    else:
        # Same rows as X_blend_train / X_blend_valid, but from non-gap X_train
        train_frame = X_train.loc[X_blend_train.index].reset_index(drop=True)
        valid_frame = X_train.loc[X_blend_valid.index].reset_index(drop=True)
        y_train_part = y_occ_train.loc[X_blend_train.index].reset_index(drop=True)
        y_valid_part = y_occ_train.loc[X_blend_valid.index].reset_index(drop=True)

        _, valid_pred = fit_predict_occupancy(
            model_name=model_name,
            X_train=train_frame,
            y_train=y_train_part,
            X_test=valid_frame,
            feature_columns=features,
            logit_target=logit_target,
        )

        validation_predictions[variant] = valid_pred
        continue

    y_train_part = y_occ_train.loc[X_blend_train.index].reset_index(drop=True)
    y_valid_part = y_occ_train.loc[X_blend_valid.index].reset_index(drop=True)

    _, valid_pred = fit_predict_occupancy(
        model_name=model_name,
        X_train=train_frame.reset_index(drop=True),
        y_train=y_train_part,
        X_test=valid_frame.reset_index(drop=True),
        feature_columns=features,
        logit_target=logit_target,
    )

    validation_predictions[variant] = valid_pred


# Add validation prediction for the existing H blend
blend_tuning_inner = tune_xgb_hgb_blend(
    X_blend_train.reset_index(drop=True),
    y_occ_train.loc[X_blend_train.index].reset_index(drop=True),
    price_gap_features,
)

_, validation_predictions["H_xgb_hgb_blend_plus_price_gap"] = predict_xgb_hgb_blend(
    X_blend_train.reset_index(drop=True),
    y_occ_train.loc[X_blend_train.index].reset_index(drop=True),
    X_blend_valid.reset_index(drop=True),
    price_gap_features,
    blend_tuning_inner["best_weight_xgb"],
)

# Use the already-known full test ranking to choose top candidates
top3_variants = ranked_overall_df["model_variant"].head(3).tolist()
top2_variants = ranked_overall_df["model_variant"].head(2).tolist()

top2_defs = [
    {"variant": variant, "valid_pred": validation_predictions[variant]}
    for variant in top2_variants
]

top3_defs = [
    {"variant": variant, "valid_pred": validation_predictions[variant]}
    for variant in top3_variants
]

top2_blend_df, top2_best = tune_blend_weights_on_validation(
    top2_defs,
    y_occ_train.loc[X_blend_valid.index].reset_index(drop=True),
    y_occ_train.loc[X_blend_valid.index].reset_index(drop=True),
)

top3_blend_df, top3_best = tune_blend_weights_on_validation(
    top3_defs,
    y_occ_train.loc[X_blend_valid.index].reset_index(drop=True),
    y_occ_train.loc[X_blend_valid.index].reset_index(drop=True),
)

dynamic_blend_validation_df = pd.concat(
    [top2_blend_df, top3_blend_df],
    ignore_index=True,
).sort_values(["rmse", "mae", "r2"], ascending=[True, True, False])

print("Top 2:", top2_variants)
print("Top 3:", top3_variants)
print("Best top-2 blend:", top2_best)
print("Best top-3 blend:", top3_best)

dynamic_blend_validation_df.head(10)

Top 2: ['H_xgb_hgb_blend_plus_price_gap', 'G_xgb_logit_plus_price_gap']
Top 3: ['H_xgb_hgb_blend_plus_price_gap', 'G_xgb_logit_plus_price_gap', 'A_hgb_baseline']
Best top-2 blend: {'blend_name': 'blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_logit_plus_price_gap', 'model_count': 2, 'components': ['H_xgb_hgb_blend_plus_price_gap', 'G_xgb_logit_plus_price_gap'], 'weights': {'H_xgb_hgb_blend_plus_price_gap': 0.4, 'G_xgb_logit_plus_price_gap': 0.6}, 'r2': 0.745792538598826, 'rmse': 0.08185986849475844, 'mae': 0.05772720342324371}
Best top-3 blend: {'blend_name': 'blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_logit_plus_price_gap__A_hgb_baseline', 'model_count': 3, 'components': ['H_xgb_hgb_blend_plus_price_gap', 'G_xgb_logit_plus_price_gap', 'A_hgb_baseline'], 'weights': {'H_xgb_hgb_blend_plus_price_gap': 0.2, 'G_xgb_logit_plus_price_gap': 0.6, 'A_hgb_baseline': 0.20000000000000007}, 'r2': 0.7461506766037929, 'rmse': 0.08180218438331478, 'mae': 0.057541767597746915}


,blend_name,model_count,components,weights,r2,rmse,mae
111,blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_lo...,3,"[H_xgb_hgb_blend_plus_price_gap, G_xgb_logit_p...","{'H_xgb_hgb_blend_plus_price_gap': 0.2, 'G_xgb...",0.746151,0.081802,0.057542
94,blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_lo...,3,"[H_xgb_hgb_blend_plus_price_gap, G_xgb_logit_p...","{'H_xgb_hgb_blend_plus_price_gap': 0.15, 'G_xg...",0.746145,0.081803,0.057525
128,blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_lo...,3,"[H_xgb_hgb_blend_plus_price_gap, G_xgb_logit_p...","{'H_xgb_hgb_blend_plus_price_gap': 0.25, 'G_xg...",0.746130,0.081805,0.057579
93,blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_lo...,3,"[H_xgb_hgb_blend_plus_price_gap, G_xgb_logit_p...","{'H_xgb_hgb_blend_plus_price_gap': 0.15, 'G_xg...",0.746125,0.081806,0.057511
75,blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_lo...,3,"[H_xgb_hgb_blend_plus_price_gap, G_xgb_logit_p...","{'H_xgb_hgb_blend_plus_price_gap': 0.1, 'G_xgb...",0.746122,0.081807,0.057494
112,blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_lo...,3,"[H_xgb_hgb_blend_plus_price_gap, G_xgb_logit_p...","{'H_xgb_hgb_blend_plus_price_gap': 0.2, 'G_xgb...",0.746121,0.081807,0.057561
127,blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_lo...,3,"[H_xgb_hgb_blend_plus_price_gap, G_xgb_logit_p...","{'H_xgb_hgb_blend_plus_price_gap': 0.25, 'G_xg...",0.746117,0.081808,0.057562
143,blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_lo...,3,"[H_xgb_hgb_blend_plus_price_gap, G_xgb_logit_p...","{'H_xgb_hgb_blend_plus_price_gap': 0.3, 'G_xgb...",0.746100,0.081810,0.057600
76,blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_lo...,3,"[H_xgb_hgb_blend_plus_price_gap, G_xgb_logit_p...","{'H_xgb_hgb_blend_plus_price_gap': 0.1, 'G_xgb...",0.746100,0.081810,0.057510
110,blend_H_xgb_hgb_blend_plus_price_gap__G_xgb_lo...,3,"[H_xgb_hgb_blend_plus_price_gap, G_xgb_logit_p...","{'H_xgb_hgb_blend_plus_price_gap': 0.2, 'G_xgb...",0.746088,0.081812,0.057530


In [21]:
best_dynamic_blend = (
    pd.DataFrame([top2_best, top3_best])
    .sort_values(["rmse", "mae", "r2"], ascending=[True, True, False])
    .iloc[0]
    .to_dict()
)

dynamic_blend_components = best_dynamic_blend["components"]
dynamic_blend_weights = best_dynamic_blend["weights"]

dynamic_blend_test_pred = np.zeros(len(y_occ_test), dtype=float)

for component in dynamic_blend_components:
    dynamic_blend_test_pred += dynamic_blend_weights[component] * predictions[component]

dynamic_blend_test_pred = np.clip(dynamic_blend_test_pred, 0, 1)

predictions["I_dynamic_top_model_blend"] = dynamic_blend_test_pred

model_registry["I_dynamic_top_model_blend"] = {
    "model_type": "dynamic_top_model_blend",
    "base_model": "weighted_blend_of_top_ranked_candidates",
    "components": dynamic_blend_components,
    "weights": dynamic_blend_weights,
    "feature_columns": None,
    "uses_price_gap": any(model_registry[c]["uses_price_gap"] for c in dynamic_blend_components),
    "logit_target": False,
}

print("Dynamic blend components:", dynamic_blend_components)
print("Dynamic blend weights:", dynamic_blend_weights)
print("Dynamic blend test metrics:", occupancy_metric_bundle(y_occ_test, dynamic_blend_test_pred))

Dynamic blend components: ['H_xgb_hgb_blend_plus_price_gap', 'G_xgb_logit_plus_price_gap', 'A_hgb_baseline']
Dynamic blend weights: {'H_xgb_hgb_blend_plus_price_gap': 0.2, 'G_xgb_logit_plus_price_gap': 0.6, 'A_hgb_baseline': 0.20000000000000007}
Dynamic blend test metrics: {'r2': 0.7381844524896007, 'rmse': 0.08347654287324272, 'mae': 0.05854895708500599}


In [22]:
overall_df_raw = pd.DataFrame(
    [
        {
            "model_variant": model_name,
            **occupancy_metric_bundle(y_occ_test, pred),
        }
        for model_name, pred in predictions.items()
    ]
)

selection = choose_best_occupancy_model(overall_df_raw)

selected_model_name = selection["selected_model"]
ranked_overall_df = selection["ranked_metrics"]

print("Selected model:", selected_model_name)
print("R2:", selection["selected_r2"])
print("RMSE:", selection["selected_rmse"])
print("MAE:", selection["selected_mae"])

ranked_overall_df

Selected model: I_dynamic_top_model_blend
R2: 0.7381844524896007
RMSE: 0.08347654287324272
MAE: 0.05854895708500599


,model_variant,r2,rmse,mae
0,I_dynamic_top_model_blend,0.738184,0.083477,0.058549
1,H_xgb_hgb_blend_plus_price_gap,0.735411,0.083917,0.059229
2,G_xgb_logit_plus_price_gap,0.735255,0.083942,0.058926
3,A_hgb_baseline,0.734456,0.084069,0.059150
4,D_hgb_plus_price_gap,0.733111,0.084282,0.059279
5,E_xgb_plus_price_gap,0.731436,0.084546,0.059921
6,B_xgb_baseline,0.729949,0.084779,0.060141
7,F_catboost_plus_price_gap,0.726001,0.085397,0.061769
8,C_catboost_baseline,0.722559,0.085931,0.062417
